# MCP package import smoke tests

This notebook tests whether the current Jupyter kernel can import and inspect:

- PyPI distribution `rope-mcp-server`, import package `rope_mcp_server`
- PyPI distribution `graphifyy`, import/CLI package `graphify`

It avoids assuming stable internal APIs. It first verifies the interpreter, then installs with captured output, then imports and introspects modules.

In [ ]:
import sys, os, site, subprocess, importlib, importlib.util, pkgutil, json, textwrap, pathlib, traceback

print("Kernel executable:", sys.executable)
print("Python version:", sys.version)
print("sys.prefix:", sys.prefix)
print("base_prefix:", getattr(sys, "base_prefix", None))
print("site-packages:")
for p in site.getsitepackages() if hasattr(site, "getsitepackages") else []:
    print(" ", p)
print("user site:", site.getusersitepackages() if hasattr(site, "getusersitepackages") else None)

## Install packages

Set `INSTALL_MODE`:

- `"pypi"`: install from PyPI
- `"github"`: install directly from the GitHub repos
- `"skip"`: skip installation and only test imports

The install cell captures stdout/stderr so the real failure reason is visible.

In [ ]:
INSTALL_MODE = "pypi"   # "pypi", "github", or "skip"
UPGRADE = True

PYPI_PACKAGES = [
    "rope-mcp-server",
    "graphifyy",
]

GITHUB_PACKAGES = [
    "git+https://github.com/krystofbe/rope-mcp-server.git",
    "git+https://github.com/safishamsi/graphify.git",
]

def run(cmd):
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("\n--- STDOUT ---")
    print(proc.stdout[-4000:] if proc.stdout else "")
    print("\n--- STDERR ---")
    print(proc.stderr[-8000:] if proc.stderr else "")
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc

if INSTALL_MODE == "skip":
    print("Skipping installation.")
else:
    packages = PYPI_PACKAGES if INSTALL_MODE == "pypi" else GITHUB_PACKAGES
    cmd = [sys.executable, "-m", "pip", "install"]
    if UPGRADE:
        cmd.append("-U")
    cmd.extend(packages)
    try:
        run(cmd)
    except Exception as e:
        print("\nInstall failed.")
        print("Common causes:")
        print("  1. This notebook is attached to the wrong kernel/interpreter.")
        print("  2. Python version is too new for one dependency, especially 3.13.")
        print("  3. pip cannot build a native dependency.")
        print("  4. PyPI package metadata changed.")
        print("\nTry either:")
        print('  - set INSTALL_MODE = "github" and rerun this cell')
        print("  - create/use a Python 3.12 uv kernel:")
        print("      uv venv --python 3.12")
        print("      uv pip install ipykernel")
        print("      uv run python -m ipykernel install --user --name mcp-test --display-name 'mcp-test (uv py3.12)'")
        raise

In [ ]:
# Verify installed distributions without importing their packages
import importlib.metadata as md

for dist_name in ["rope-mcp-server", "graphifyy"]:
    print("=" * 80)
    print("Distribution:", dist_name)
    try:
        dist = md.distribution(dist_name)
        print("version:", dist.version)
        print("location:", dist.locate_file(""))
        top = dist.read_text("top_level.txt")
        if top:
            print("top_level.txt:", top.strip())
        eps = list(dist.entry_points)
        if eps:
            print("entry points:")
            for ep in eps:
                print(" ", ep.group, ep.name, "=", ep.value)
    except Exception as e:
        print("NOT FOUND:", repr(e))

## Import and module discovery smoke tests

In [ ]:
CANDIDATE_IMPORTS = [
    "rope_mcp_server",
    "rope_mcp_server.server",
    "rope_mcp_server.main",
    "graphify",
    "graphify.cli",
    "graphify.main",
]

def import_report(name):
    print("=" * 80)
    print("Import:", name)
    spec = importlib.util.find_spec(name)
    print("spec:", spec)
    if spec is None:
        return None
    try:
        mod = importlib.import_module(name)
        print("OK:", mod)
        print("file:", getattr(mod, "__file__", None))
        public = [x for x in dir(mod) if not x.startswith("_")]
        print("public names:", public[:80])
        return mod
    except Exception:
        print("FAILED:")
        traceback.print_exc(limit=8)
        return None

mods = {name: import_report(name) for name in CANDIDATE_IMPORTS}

In [ ]:
# Walk top-level submodules safely. This only lists modules; it does not import every one.
for root_name in ["rope_mcp_server", "graphify"]:
    print("=" * 80)
    print("Submodules under:", root_name)
    root = importlib.import_module(root_name)
    path = getattr(root, "__path__", None)
    if path is None:
        print("Not a package or no __path__.")
        continue
    for i, m in enumerate(pkgutil.walk_packages(path, prefix=root_name + ".")):
        print(m.name, "(pkg)" if m.ispkg else "")
        if i >= 200:
            print("... truncated ...")
            break

## Find MCP-style server objects/functions

This searches imported modules for names that look like MCP server entrypoints or tool registrations.

In [ ]:
KEYWORDS = ("server", "mcp", "tool", "rename", "rope", "graph", "analy", "refactor", "run", "main")

for name, mod in mods.items():
    if mod is None:
        continue
    print("=" * 80)
    print(name)
    hits = []
    for attr in dir(mod):
        lower = attr.lower()
        if any(k in lower for k in KEYWORDS):
            try:
                obj = getattr(mod, attr)
                hits.append((attr, type(obj).__name__, repr(obj)[:140]))
            except Exception as e:
                hits.append((attr, "ERROR", repr(e)))
    for row in hits[:100]:
        print(row)

## Optional: import likely submodules and inspect callable signatures

Edit `SUBMODULES_TO_PROBE` after reviewing the submodule list above.

In [ ]:
import inspect

SUBMODULES_TO_PROBE = [
    "rope_mcp_server.server",
    "rope_mcp_server.main",
    "graphify.cli",
    "graphify.main",
]

for mod_name in SUBMODULES_TO_PROBE:
    print("=" * 80)
    print("Probe:", mod_name)
    try:
        mod = importlib.import_module(mod_name)
    except Exception:
        traceback.print_exc(limit=8)
        continue

    for attr in dir(mod):
        if attr.startswith("_"):
            continue
        try:
            obj = getattr(mod, attr)
        except Exception:
            continue
        if inspect.isfunction(obj) or inspect.isclass(obj):
            try:
                sig = str(inspect.signature(obj))
            except Exception:
                sig = "(signature unavailable)"
            print(f"{attr}: {type(obj).__name__}{sig}")

## Optional: CLI smoke tests

These commands only ask for help/version where available. They should not modify a repo.

In [ ]:
CLI_COMMANDS = [
    ["graphify", "--help"],
    [sys.executable, "-m", "graphify", "--help"],
    [sys.executable, "-m", "rope_mcp_server", "--help"],
]

for cmd in CLI_COMMANDS:
    print("=" * 80)
    try:
        proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=20)
        print("$", " ".join(cmd))
        print("exit:", proc.returncode)
        print("--- STDOUT ---")
        print(proc.stdout[:3000])
        print("--- STDERR ---")
        print(proc.stderr[:3000])
    except FileNotFoundError as e:
        print("missing executable:", cmd[0], e)
    except subprocess.TimeoutExpired:
        print("timed out:", cmd)